# NSE Stock Analyser
**Run every cell top to bottom. On phone: tap the ▶ button on each cell.**

- Cell 1: Install (run once per session)
- Cell 2: Mount Google Drive (saves predictions across sessions)
- Cell 3: General analysis — daily view, NIFTY 50
- Cell 3b: General — strict mode (highest conviction only)
- Cell 4: Intraday analysis — 1h candles, swing picks
- Cell 5: Commodity ETFs — Gold & Silver analysis
- Cell 6: Record predictions
- Cell 7: Evaluate predictions
- Cell 8: Sector filter

In [ ]:
# ── Cell 1: Setup (run once per session) ─────────────────────────────────────
import os

REPO_URL = "https://github.com/anikgupta1970/stockFnF"
REPO_DIR = "stockFnF"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
else:
    os.system(f"cd {REPO_DIR} && git pull")

os.chdir(REPO_DIR)
os.system("pip install -q -r requirements.txt")
print("✅ Setup complete. Working directory:", os.getcwd())

In [ ]:
# ── Cell 2: Mount Google Drive (optional but recommended) ─────────────────────
# This saves your predictions file permanently across Colab sessions.
# Skip this cell if you don't need to save predictions.

from google.colab import drive
drive.mount('/content/drive')

PRED_DIR = "/content/drive/MyDrive/StockPredictions"
os.makedirs(PRED_DIR, exist_ok=True)

GENERAL_FILE   = f"{PRED_DIR}/general_preds.json"
INTRADAY_FILE  = f"{PRED_DIR}/intraday_preds.json"
COMMODITY_FILE = f"{PRED_DIR}/commodity_preds.json"

print(f"✅ Drive mounted. Predictions will be saved to {PRED_DIR}")

In [ ]:
# ── Cell 3: General Analysis — NIFTY 50 (daily candles) ───────────────────────
# Shows top 20 NIFTY 50 stocks ranked by buy probability.
# Includes score, entry/stop/target levels, regime, fundamentals, backtest.

!python3 main.py general --index nifty50 --top 20 --no-market-filter

In [ ]:
# ── Cell 3b: General — Strict Mode (highest conviction only) ──────────────────
# Only shows stocks where ALL 4 conditions agree:
# Score ≥ 70 + MACD bullish + Golden cross + RSI < 65

!python3 main.py general --index nifty50 --strict --no-market-filter

In [ ]:
# ── Cell 4: Intraday / Swing Analysis (1h candles) ────────────────────────────
# Best run before market opens (9:00-9:15 AM) or mid-session (11-12 PM).
# Shows 1-5 day swing trade setups.

!python3 main.py intraday --index nifty50 --top 10

In [ ]:
# ── Cell 5: Commodity ETFs — Gold & Silver ────────────────────────────────────
# Analyses Gold and Silver ETFs traded on NSE.
# Note: --no-market-filter is used because gold often moves opposite to equities.

print("=== All Commodities (Gold + Silver) ===")
!python3 main.py general --index commodities --no-market-filter

print("\n=== Gold ETFs only ===")
!python3 main.py general --index commodities --sector Gold --no-market-filter

print("\n=== Silver ETFs only ===")
!python3 main.py general --index commodities --sector Silver --no-market-filter

In [ ]:
# ── Cell 6: Record Predictions ────────────────────────────────────────────────
# Run this after analysis cells to save today's picks for later evaluation.
# Requires Cell 2 (Drive mount) to persist across sessions.

# Check if Drive was mounted, else save locally
if 'GENERAL_FILE' not in dir():
    GENERAL_FILE   = "general_preds.json"
    INTRADAY_FILE  = "intraday_preds.json"
    COMMODITY_FILE = "commodity_preds.json"
    print("⚠️  Drive not mounted — predictions saved locally (will be lost on reset)")
    print("   Run Cell 2 first to save permanently to Google Drive.")

print("\nRecording general picks...")
!python3 predict.py record --mode general  --top 10 --threshold 60 --no-market-filter --file "{GENERAL_FILE}"

print("\nRecording intraday picks...")
!python3 predict.py record --mode intraday --top 10 --threshold 60 --file "{INTRADAY_FILE}"

print("\nRecording commodity picks...")
!python3 predict.py record --mode general --index commodities --threshold 45 --no-market-filter --file "{COMMODITY_FILE}"

In [ ]:
# ── Cell 7: Evaluate Predictions ─────────────────────────────────────────────
# Run this any time after recording to check if stop/target was hit.

if 'GENERAL_FILE' not in dir():
    GENERAL_FILE   = "general_preds.json"
    INTRADAY_FILE  = "intraday_preds.json"
    COMMODITY_FILE = "commodity_preds.json"

print("General predictions:")
!python3 predict.py evaluate --file "{GENERAL_FILE}"

print("\nIntraday predictions:")
!python3 predict.py evaluate --file "{INTRADAY_FILE}"

print("\nCommodity predictions:")
!python3 predict.py evaluate --file "{COMMODITY_FILE}"

In [ ]:
# ── Cell 8: Sector Filter (optional) ─────────────────────────────────────────
# Change SECTOR below to filter results.
# Stocks:     Banking, IT, Pharma, Auto, FMCG, Energy, Metals, Infrastructure,
#             Consumer, Financial Services, Insurance, Telecom, Cement, Chemicals
# Commodities: Gold, Silver

SECTOR = "IT"   # ← change this

!python3 main.py general --index nifty50 --sector "{SECTOR}" --no-market-filter